# STEP 1: SETUP WITH GROQ

In [ ]:
#Install libraries
!pip install langchain langchain-community langchain-groq langsmith

In [ ]:

#Setup environment
import os

# Groq API Key
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

# LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-System"

In [ ]:
#Test model (Groq)
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

response = llm.invoke("Say hello")
print(response.content)

# STEP 2: Job Description + 3 Resumes

In [ ]:
job_description = """
We are looking for a Data Scientist with the following skills:

- Strong knowledge of Python
- Experience with Machine Learning (Scikit-learn, TensorFlow, or PyTorch)
- Data Analysis using Pandas and NumPy
- SQL and database knowledge
- Data Visualization (Matplotlib, Seaborn, Tableau)
- Experience with real-world datasets
- Good understanding of statistics

Preferred:
- Deep Learning
- NLP experience
- Deployment knowledge (Flask, FastAPI)
"""

In [ ]:
resume_strong = """
Name: Alice

Skills:
Python, Machine Learning, Deep Learning, NLP, TensorFlow, PyTorch, Pandas, NumPy, SQL, Tableau

Experience:
3 years as Data Scientist working on real-world ML projects including NLP models and predictive analytics.

Projects:
Built ML models using TensorFlow and deployed using Flask.
"""

In [ ]:
resume_average = """
Name: Bob

Skills:
Python, Pandas, NumPy, SQL, Matplotlib

Experience:
1 year as Data Analyst working on data cleaning and visualization.

Projects:
Performed data analysis on sales dataset using Python.
"""

In [ ]:
resume_weak = """
Name: Charlie

Skills:
MS Excel, PowerPoint, Basic Python

Experience:
Fresher with no real-world project experience.

Projects:
Created basic Excel reports.
"""

# STEP 3: Skill Extraction (LangChain + PromptTemplate)

In [ ]:
from langchain_core.prompts import PromptTemplate

extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an AI system that extracts structured information from resumes.

Extract the following:
1. Skills
2. Experience
3. Tools/Technologies

Rules:
- Only extract what is present
- Do NOT assume anything
- Keep output clean

Output format:
Skills: ...
Experience: ...
Tools: ...

Resume:
{resume}
"""
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

extract_chain = extract_prompt | llm | StrOutputParser()

In [ ]:
result = extract_chain.invoke({"resume": resume_strong})
print(result)

# STEP 4: Matching Logic (Resume vs Job Description)

In [ ]:
match_prompt = PromptTemplate(
    input_variables=["resume", "job_description"],
    template="""
You are an AI system that compares a resume with a job description.

Tasks:
1. Identify matching skills
2. Identify missing skills
3. Evaluate how well the resume fits the job

Rules:
- Do NOT assume skills
- Only use given data
- Be accurate

Output format:
Matching Skills: ...
Missing Skills: ...
Summary: ...

Job Description:
{job_description}

Resume:
{resume}
"""
)

In [ ]:
match_chain = match_prompt | llm | StrOutputParser()

In [ ]:
match_result = match_chain.invoke({
    "resume": resume_strong,
    "job_description": job_description
})

print(match_result)

# STEP 5: Scoring System (0–100)

In [ ]:
score_prompt = PromptTemplate(
    input_variables=["resume", "job_description"],
    template="""
You are an AI system that scores resumes based on job fit.

Task:
- Assign a score from 0 to 100

Scoring Rules:
- 90–100 → Excellent match (almost all skills match)
- 70–89 → Good match (most skills match)
- 50–69 → Average match
- Below 50 → Weak match

Strict Rules:
- Do NOT assume missing skills
- Penalize missing important skills
- Be realistic (avoid giving everyone high score)

Output format:
Score: <number>
Reason: <short reason>

Job Description:
{job_description}

Resume:
{resume}
"""
)

In [ ]:
score_chain = score_prompt | llm | StrOutputParser()

In [ ]:
score_result = score_chain.invoke({
    "resume": resume_strong,
    "job_description": job_description
})

print(score_result)

# STEP 6: Explanation Generator (WHY this score?)

In [ ]:
explain_prompt = PromptTemplate(
    input_variables=["resume", "job_description", "score"],
    template="""
You are an AI assistant explaining resume evaluation results.

Task:
Explain why the candidate received the given score.

Include:
- Strengths (what matched well)
- Weaknesses (what is missing)
- Final justification

Rules:
- Be clear and concise
- Do NOT assume anything
- Base only on given data

Output format:
Explanation: ...

Score: {score}

Job Description:
{job_description}

Resume:
{resume}
"""
)

In [ ]:
explain_chain = explain_prompt | llm | StrOutputParser()

In [ ]:
explain_result = explain_chain.invoke({
    "resume": resume_strong,
    "job_description": job_description,
    "score": score_result
})

print(explain_result)

# STEP 7: Combine Full Pipeline + Tracing

In [ ]:
def run_pipeline(resume):
    # Step 1: Extraction
    extracted = extract_chain.invoke({"resume": resume})

    # Step 2: Matching
    match = match_chain.invoke({
        "resume": resume,
        "job_description": job_description
    })

    # Step 3: Scoring
    score = score_chain.invoke({
        "resume": resume,
        "job_description": job_description
    })

    # Step 4: Explanation
    explanation = explain_chain.invoke({
        "resume": resume,
        "job_description": job_description,
        "score": score
    })

    return {
        "extracted": extracted,
        "match": match,
        "score": score,
        "explanation": explanation
    }

In [ ]:
print("=== STRONG CANDIDATE ===")
strong_result = run_pipeline(resume_strong)
print(strong_result)

print("\n=== AVERAGE CANDIDATE ===")
avg_result = run_pipeline(resume_average)
print(avg_result)

print("\n=== WEAK CANDIDATE ===")
weak_result = run_pipeline(resume_weak)
print(weak_result)